In [ ]:
import os
from azure.ai.ml import MLClient, command, Input
from azure.ai.ml.entities import Environment
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id=os.getenv("SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("RESOURCE_GROUP"),
    workspace_name=os.getenv("WORKSPACE")
)

env = Environment(
    name="asr-compression-env",
    image="mcr.microsoft.com/azureml/openmpi5.0-cuda12.6-ubuntu24.04:20260303.v5",
    conda_file="./azure_env/conda.yml"
)

inputs = {
    "features": Input(
        type="uri_folder",
        path="azureml:common_voice_german-preprocessed:3",
        mode="ro_mount"
    )
}
environment_variables = {
    "DATASET_MOUNT_BLOCK_BASED_CACHE_ENABLED": "true",
    "DATASET_MOUNT_BLOCK_FILE_CACHE_ENABLED": "true",
    "DATASET_MOUNT_MEMORY_CACHE_SIZE": "0",
    "DATASET_MOUNT_READ_BUFFER_BLOCK_COUNT": "80"
}

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


# 1. baseline -> eval

In [ ]:
MODEL = "usefulsensors/moonshine-tiny"
RAW_MODEL = MODEL.split("/")[1]

job = command(
    code=".",
    command=f"python -m scripts.onnx_eval_pipeline --model-path {MODEL} --dataset-path ${{inputs.features}}/{RAW_MODEL} --output-dir outputs --quantization none",
    inputs=inputs,
    environment_variables=environment_variables,
    environment=env,
    compute="big-cpu-cluster",
    experiment_name="philipp-compression-framework",
    display_name=f"new-{MODEL}-base-eval"
)

returned_job = ml_client.jobs.create_or_update(job)

# 2. adalora -> eval

In [ ]:
MODEL = "usefulsensors/moonshine-tiny"
RAW_MODEL = MODEL.split("/")[1]

job = command(
    code=".",
    command=f"python -m scripts.adalora_train_and_evaluate --model-name {MODEL} --dataset-path ${{inputs.features}}/{RAW_MODEL} --output-dir outputs --skip-eval",
    inputs=inputs,
    environment_variables=environment_variables,
    environment=env,
    compute="gpucluster-pr-1",
    experiment_name="philipp-compression-framework",
    display_name=f"new-{MODEL}-adalora-eval"
)

returned_job = ml_client.jobs.create_or_update(job)

# 3. base -> int4 -> eval

In [ ]:
MODEL = "usefulsensors/moonshine-tiny"
RAW_MODEL = MODEL.split("/")[1]

job = command(
    code=".",
    command=f"python -m scripts.onnx_eval_pipeline --model-name {MODEL} --dataset-path ${{inputs.features}}/{RAW_MODEL} --output-dir outputs --quantization int4",
    inputs=inputs,
    environment_variables=environment_variables,
    environment=env,
    compute="big-cpu-cluster",
    experiment_name="philipp-compression-framework",
    display_name=f"new-{MODEL}-base-int4-eval"
)

returned_job = ml_client.jobs.create_or_update(job)

# 4. base -> adalora -> int4 -> eval

In [ ]:
MODEL = "usefulsensors/moonshine-tiny"
RAW_MODEL = MODEL.split("/")[1]

job = command(
    code=".",
    command=f"python -m scripts.onnx_eval_pipeline --model-path ./results/ --dataset-path ${{inputs.features}}/{RAW_MODEL} --output-dir outputs --quantization int4",
    inputs=inputs,
    environment_variables=environment_variables,
    environment=env,
    compute="gpucluster-pr-1",
    experiment_name="philipp-compression-framework",
    display_name=f"new-{MODEL}-base-adalora-int4-eval"
)

returned_job = ml_client.jobs.create_or_update(job)

# base -> int4 -> adalora -> eval

python scripts/adalora_train_and_evaluate.py \
  --model-name $MODEL \
  --dataset-path $DATA \
  --output-dir runs/int4_adalora \
  --quantization int4 \
  --skip-eval

python scripts/merge_adapter.py \
  --base-model $MODEL \
  --adapter-path runs/int4_adalora \
  --output-dir runs/int4_adalora_merged

python scripts/onnx_eval_pipeline.py \
  --model-path runs/int4_adalora_merged \
  --dataset-path $DATA \
  --output-dir runs/int4_adalora_onnx \
  --quantization none


# 7. base -> adalora -> int8 -> eval

In [ ]:
MODEL = "usefulsensors/moonshine-tiny"
RAW_MODEL = MODEL.split("/")[1]

job = command(
    code=".",
    command=f"python -m scripts.onnx_eval_pipeline --model-path ./results/ --dataset-path ${{inputs.features}}/{RAW_MODEL} --output-dir outputs --quantization int8",
    inputs=inputs,
    environment_variables=environment_variables,
    environment=env,
    compute="gpucluster-pr-1",
    experiment_name="philipp-compression-framework",
    display_name=f"new-{MODEL}-base-adalora-int8-eval"
)

returned_job = ml_client.jobs.create_or_update(job)

# base -> int8 -> adalora -> eval

python scripts/adalora_train_and_evaluate.py \
  --model-name $MODEL \
  --dataset-path $DATA \
  --output-dir runs/int8_adalora \
  --quantization int8 \
  --skip-eval

python scripts/merge_adapter.py \
  --base-model $MODEL \
  --adapter-path runs/int8_adalora \
  --output-dir runs/int8_adalora_merged

python scripts/onnx_eval_pipeline.py \
  --model-path runs/int8_adalora_merged \
  --dataset-path $DATA \
  --output-dir runs/int8_adalora_onnx \
  --quantization none
